# DPDP-Audit: Qwen2.5-7B-Instruct on vLLM (Kaggle, 2x T4)

Interactive version of `kaggle/b1b2/run.py`: serves the reasoning model through vLLM's
OpenAI-compatible API on one T4, checks that it returns token log-probabilities, and runs
the experiment scripts against it. For unattended full runs use `scripts/kaggle_push.sh`.

T4 GPUs have no FlashAttention; newer vLLM releases fall back to FlexAttention, which is
about 10x slower. vLLM 0.10.2 with the V0 engine and xFormers is the tested setup.

Settings: *Accelerator = GPU T4 x2*, *Internet = on*. Add the repository as a Kaggle dataset
named `dpdp-audit` (or set `REPO_URL` below to a git URL).

In [ ]:
!pip install -q "vllm==0.10.2" "transformers==4.56.2"

In [ ]:
import subprocess, time, requests, os
MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", VLLM_USE_V1="0", VLLM_ATTENTION_BACKEND="XFORMERS")
server = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL,
     "--quantization", "awq", "--dtype", "half", "--max-model-len", "12288",
     "--gpu-memory-utilization", "0.92", "--max-num-seqs", "16", "--port", "8000"],
    stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT, env=env)
for _ in range(120):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            print("server ready"); break
    except Exception:
        time.sleep(10)
else:
    print(open("vllm.log").read()[-3000:])

## Stage 0 check: the model answers a test prompt with log-probabilities

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")
r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "Reply with the word OK."}],
                                   max_tokens=5, logprobs=True, temperature=0)
print(r.choices[0].message.content, [round(t.logprob, 3) for t in r.choices[0].logprobs.content])

## Run experiments against the local server

In [ ]:
REPO_URL = ""  # e.g. a git URL; leave empty to use the Kaggle dataset copy
if REPO_URL:
    !git clone -q {REPO_URL} /kaggle/working/dpdp-audit
else:
    !cp -r /kaggle/input/dpdp-audit /kaggle/working/dpdp-audit
%cd /kaggle/working/dpdp-audit
!pip install -q -e . python-dotenv faiss-cpu sentence-transformers google-genai "transformers==4.56.2"
os.environ.update(LLM_BACKEND="openai", OPENAI_BASE_URL="http://localhost:8000/v1", OPENAI_MODEL=MODEL)

In [ ]:
!python scripts/run_baselines.py --systems B1 B2 --seeds 0 1 2 --workers 8 --redo-failed
!python scripts/score_runs.py --systems B1 B2

In [ ]:
# Keep results: download results.tar.gz from the output panel and unpack it at the repository root.
!tar czf /kaggle/working/results.tar.gz results